# 02 - Train 5 Transfer Success Models with MLflow


- load the dataset
- remove leakage columns
- validate with cross-validation
- tune hyperparameters with `GridSearchCV`
- log all runs to MLflow
- compare training and test performance


## 1) Imports


In [19]:
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier


## 2) Load dataset


In [20]:
ROOT = Path.cwd().resolve().parent
DATA_PATH = ROOT / 'data' / 'processed' / 'transfer_modeling_dataset.csv'

df = pd.read_csv(DATA_PATH)
print('Dataset shape:', df.shape)
df.head()
df.info()
df.describe()

Dataset shape: (3581, 44)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3581 entries, 0 to 3580
Data columns (total 44 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   transfer_row_id               3581 non-null   int64  
 1   player_id                     3581 non-null   int64  
 2   transfer_date                 3581 non-null   object 
 3   transfer_season               3581 non-null   object 
 4   from_club_id                  3581 non-null   int64  
 5   to_club_id                    3581 non-null   int64  
 6   from_club_name                3581 non-null   object 
 7   to_club_name                  3581 non-null   object 
 8   transfer_fee_eur              3581 non-null   float64
 9   market_value_at_transfer_eur  3581 non-null   float64
 10  age_at_transfer               3581 non-null   float64
 11  primary_position              3581 non-null   object 
 12  secondary_position            3581 n

,transfer_row_id,player_id,from_club_id,to_club_id,transfer_fee_eur,market_value_at_transfer_eur,age_at_transfer,height_cm,pre_minutes_total,pre_goals_total,...,has_api_prev_standing,has_api_current_form,post_minutes_avg_per_season,post_goals_per90_avg,post_assists_per90_avg,market_value_end_window_eur,criteria_minutes,criteria_value,criteria_position_kpi,transfer_success
count,3581.000000,3.581000e+03,3581.000000,3581.000000,3.581000e+03,3.581000e+03,3581.000000,3581.000000,3581.000000,3581.000000,...,3581.0,3581.0,3581.000000,3581.000000,3581.000000,3.581000e+03,3581.000000,3581.000000,3581.000000,3581.000000
mean,2480.022061,4.139468e+05,2003.426696,1439.194918,3.459500e+06,1.122961e+06,25.115982,183.248255,1063.907288,1.823513,...,0.0,0.0,935.395234,0.129637,0.094308,2.351515e+06,0.496230,0.772689,0.267244,0.551522
std,1392.570395,2.071480e+05,6020.041094,3106.378897,1.013934e+07,2.457900e+05,3.751240,6.695394,1060.155506,3.604639,...,0.0,0.0,683.816736,0.179083,0.164667,1.202663e+06,0.500056,0.419154,0.442582,0.497408
min,0.000000,3.333000e+03,3.000000,3.000000,0.000000e+00,5.000000e+04,16.731007,164.500000,0.000000,0.000000,...,0.0,0.0,0.000000,0.000000,0.000000,5.000000e+04,0.000000,0.000000,0.000000,0.000000
25%,1318.000000,2.573760e+05,244.000000,244.000000,0.000000e+00,1.225000e+06,22.258726,178.000000,6.000000,0.000000,...,0.0,0.0,349.333340,0.000000,0.000000,1.200000e+06,0.000000,1.000000,0.000000,0.000000
50%,2541.000000,3.943000e+05,599.000000,583.000000,0.000000e+00,1.225000e+06,24.698153,184.000000,779.000000,0.000000,...,0.0,0.0,883.500000,0.054845,0.060753,3.000000e+06,0.000000,1.000000,0.000000,1.000000
75%,3765.000000,5.538750e+05,1132.000000,1071.000000,1.750000e+06,1.225000e+06,27.507187,188.000000,1847.000000,2.000000,...,0.0,0.0,1406.250000,0.199911,0.141003,3.450000e+06,1.000000,1.000000,1.000000,1.000000
max,4955.000000,1.141628e+06,110302.000000,23826.000000,1.270000e+08,1.225000e+06,39.852158,200.500000,4716.000000,50.000000,...,0.0,0.0,3300.000000,2.142857,3.461539,3.450000e+06,1.000000,1.000000,1.000000,1.000000


## 3) Check missing values

We do not drop rows. Missing values are handled in the preprocessing pipeline.


In [21]:
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing

Series([], dtype: int64)

## 4) Keep only features available at transfer time

This avoids target leakage from future information.


In [22]:
target_col = 'transfer_success'

leakage_cols = [
    'transfer_row_id',
    'player_id',
    'transfer_date',
    'from_club_name',
    'to_club_name',
    'post_minutes_avg_per_season',
    'post_goals_per90_avg',
    'post_assists_per90_avg',
    'market_value_end_window_eur',
    'criteria_minutes',
    'criteria_value',
    'criteria_position_kpi',
]

feature_cols = [col for col in df.columns if col not in leakage_cols + [target_col]]
X = df[feature_cols].copy()
y = df[target_col].copy()

print('Features before dropping all-null columns:', len(feature_cols))

Features before dropping all-null columns: 31


## 5) Train/test split

The holdout test set is kept separate until the end.


In [23]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)
print('Train target distribution:')
print(y_train.value_counts(normalize=True).sort_index())

Train shape: (2864, 31)
Test shape: (717, 31)
Train target distribution:
transfer_success
0    0.448324
1    0.551676
Name: proportion, dtype: float64


## 6) Preprocessing pipeline

- numeric: median imputation + scaling
- categorical: most frequent imputation + one-hot encoding


In [24]:
numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

print('Numeric features:', len(numeric_features))
print('Categorical features:', len(categorical_features))

Numeric features: 24
Categorical features: 7


## 7) Why these 5 models?

We use five models with different behavior:

- `DummyClassifier`: baseline
- `LogisticRegression`: simple linear model
- `RandomForestClassifier`: strong tree ensemble
- `SVC`: nonlinear margin-based model
- `DecisionTreeClassifier`: interpretable tree model


## 8) Define 5 models and tuning parameters

In [25]:
model_configs = {
    'baseline_dummy': {
        'model': DummyClassifier(),
        'param_grid': {
            'model__strategy': ['most_frequent', 'prior']
        },
    },
    'logistic_regression': {
        'model': LogisticRegression(max_iter=2000, random_state=42),
        'param_grid': {
            'model__C': [0.1, 1.0, 10.0],
            'model__solver': ['lbfgs'],
        },
    },
    'random_forest': {
        'model': RandomForestClassifier(random_state=42, n_jobs=1),
        'param_grid': {
            'model__n_estimators': [100, 300],
            'model__max_depth': [None, 8, 12],
            'model__min_samples_split': [2, 10],
        },
    },
    'svm_rbf': {
        'model': SVC(kernel='rbf', probability=True, random_state=42),
        'param_grid': {
            'model__C': [0.5, 1.0, 2.0],
            'model__gamma': ['scale', 'auto'],
        },
    },
    'decision_tree': {
        'model': DecisionTreeClassifier(random_state=42),
        'param_grid': {
            'model__max_depth': [4, 6, 8, None],
            'model__min_samples_split': [2, 10, 20],
        },
    },
}

model_configs

{'baseline_dummy': {'model': DummyClassifier(),
  'param_grid': {'model__strategy': ['most_frequent', 'prior']}},
 'logistic_regression': {'model': LogisticRegression(max_iter=2000, random_state=42),
  'param_grid': {'model__C': [0.1, 1.0, 10.0], 'model__solver': ['lbfgs']}},
 'random_forest': {'model': RandomForestClassifier(n_jobs=1, random_state=42),
  'param_grid': {'model__n_estimators': [100, 300],
   'model__max_depth': [None, 8, 12],
   'model__min_samples_split': [2, 10]}},
 'svm_rbf': {'model': SVC(probability=True, random_state=42),
  'param_grid': {'model__C': [0.5, 1.0, 2.0],
   'model__gamma': ['scale', 'auto']}},
 'decision_tree': {'model': DecisionTreeClassifier(random_state=42),
  'param_grid': {'model__max_depth': [4, 6, 8, None],
   'model__min_samples_split': [2, 10, 20]}}}

## 9) Validation setup

We use 3-fold stratified cross-validation on the training set.


In [ ]:
cv = StratifiedKFold(n_splits=6, shuffle=True, random_state=42)
cv

StratifiedKFold(n_splits=6, random_state=42, shuffle=True)

## 10) MLflow setup

To view the runs:

```bash
mlflow ui
```

Then open `http://127.0.0.1:5000`.


In [27]:
EXPERIMENT_NAME = 'transfer_success_simple_models'
mlflow.set_experiment(EXPERIMENT_NAME)
print('MLflow experiment:', EXPERIMENT_NAME)

MLflow experiment: transfer_success_simple_models


## 11) Helper functions

We calculate both standard metrics and business-style metrics.

Business interpretation used here:
- false positive: club thinks a transfer will succeed, but it does not
- false negative: club misses a transfer that could have succeeded

Business metrics in this notebook:
- `success_precision`: how often predicted successful transfers are really successful
- `missed_success_rate`: how often we miss true successful transfers


In [28]:
def evaluate_predictions(y_true, y_pred, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    standard_metrics = {
        'train_or_test_accuracy': accuracy_score(y_true, y_pred),
        'train_or_test_precision': precision_score(y_true, y_pred, zero_division=0),
        'train_or_test_recall': recall_score(y_true, y_pred, zero_division=0),
        'train_or_test_f1': f1_score(y_true, y_pred, zero_division=0),
        'train_or_test_roc_auc': roc_auc_score(y_true, y_proba),
    }

    business_metrics = {
        'success_precision': tp / (tp + fp) if (tp + fp) > 0 else 0.0,
        'missed_success_rate': fn / (fn + tp) if (fn + tp) > 0 else 0.0,
        'false_positive_rate': fp / (fp + tn) if (fp + tn) > 0 else 0.0,
        'captured_success_rate': tp / (tp + fn) if (tp + fn) > 0 else 0.0,
    }

    counts = {
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }

    return standard_metrics, business_metrics, counts


def prefix_metrics(metrics_dict, prefix):
    return {f'{prefix}_{k}': v for k, v in metrics_dict.items()}


def build_pipeline(model):
    return Pipeline(
        steps=[
            ('preprocessor', preprocessor),
            ('model', model),
        ]
    )


def train_tune_validate_log(model_name, model, param_grid):
    pipeline = build_pipeline(model)

    baseline_cv_scores = cross_val_score(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring='f1',
        n_jobs=1,
    )

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=cv,
        scoring='f1',
        n_jobs=1,
    )
    grid_search.fit(X_train, y_train)

    best_pipeline = grid_search.best_estimator_

    train_pred = best_pipeline.predict(X_train)
    train_proba = best_pipeline.predict_proba(X_train)[:, 1]
    train_standard, train_business, train_counts = evaluate_predictions(y_train, train_pred, train_proba)

    test_pred = best_pipeline.predict(X_test)
    test_proba = best_pipeline.predict_proba(X_test)[:, 1]
    test_standard, test_business, test_counts = evaluate_predictions(y_test, test_pred, test_proba)

    result = {
        'model_name': model_name,
        'cv_f1_mean_before_tuning': baseline_cv_scores.mean(),
        'cv_f1_std_before_tuning': baseline_cv_scores.std(),
        'best_cv_f1_after_tuning': grid_search.best_score_,
        'best_params': grid_search.best_params_,
        **prefix_metrics(train_standard, 'train'),
        **prefix_metrics(train_business, 'train'),
        **prefix_metrics(test_standard, 'test'),
        **prefix_metrics(test_business, 'test'),
        **prefix_metrics(train_counts, 'train'),
        **prefix_metrics(test_counts, 'test'),
    }

    mlflow_metrics = {
        'cv_f1_mean_before_tuning': baseline_cv_scores.mean(),
        'cv_f1_std_before_tuning': baseline_cv_scores.std(),
        'best_cv_f1_after_tuning': grid_search.best_score_,
        **prefix_metrics(train_standard, 'train'),
        **prefix_metrics(train_business, 'train'),
        **prefix_metrics(test_standard, 'test'),
        **prefix_metrics(test_business, 'test'),
    }

    with mlflow.start_run(run_name=model_name):
        mlflow.log_param('model_name', model_name)
        mlflow.log_param('target_column', target_col)
        mlflow.log_param('n_features', len(feature_cols))
        mlflow.log_param('train_rows', len(X_train))
        mlflow.log_param('test_rows', len(X_test))

        for key, value in model.get_params().items():
            mlflow.log_param(f'model__{key}', value)

        for key, value in grid_search.best_params_.items():
            mlflow.log_param(f'best__{key}', value)

        mlflow.log_metrics(mlflow_metrics)
        mlflow.sklearn.log_model(best_pipeline, name='model')

    return result, best_pipeline


## 12) Train, validate, tune, and log all 5 models


In [29]:
results = []
trained_pipelines = {}

for model_name, config in model_configs.items():
    print(f'Training: {model_name}')
    result, pipeline = train_tune_validate_log(
        model_name=model_name,
        model=config['model'],
        param_grid=config['param_grid'],
    )
    results.append(result)
    trained_pipelines[model_name] = pipeline

results_df = pd.DataFrame(results).sort_values(by='test_train_or_test_f1', ascending=False).reset_index(drop=True)
results_df

Training: baseline_dummy


2026/04/27 21:38:28 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Training: logistic_regression


2026/04/27 21:38:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Training: random_forest


...

Training: svm_rbf


2026/04/27 21:41:22 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Training: decision_tree


2026/04/27 21:41:30 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


,model_name,cv_f1_mean_before_tuning,cv_f1_std_before_tuning,best_cv_f1_after_tuning,best_params,train_train_or_test_accuracy,train_train_or_test_precision,train_train_or_test_recall,train_train_or_test_f1,train_train_or_test_roc_auc,...,test_false_positive_rate,test_captured_success_rate,train_tn,train_fp,train_fn,train_tp,test_tn,test_fp,test_fn,test_tp
0,random_forest,0.754975,0.007813,0.765248,"{'model__max_depth': None, 'model__min_samples...",1.000000,1.000000,1.000000,1.000000,1.000000,...,0.360248,0.815190,1284,0,0,1580,206,116,73,322
1,logistic_regression,0.718765,0.012823,0.723148,"{'model__C': 0.1, 'model__solver': 'lbfgs'}",0.706006,0.721489,0.760759,0.740604,0.772719,...,0.388199,0.754430,820,464,378,1202,197,125,97,298
2,svm_rbf,0.736250,0.008067,0.736250,"{'model__C': 1.0, 'model__gamma': 'scale'}",0.800279,0.814214,0.826582,0.820352,0.877383,...,0.372671,0.746835,986,298,274,1306,202,120,100,295
3,decision_tree,0.664169,0.021486,0.717518,"{'model__max_depth': 4, 'model__min_samples_sp...",0.682612,0.683233,0.791772,0.733509,0.744948,...,0.481366,0.779747,704,580,329,1251,167,155,87,308
4,baseline_dummy,0.711071,0.000367,0.711071,{'model__strategy': 'most_frequent'},0.551676,0.551676,1.000000,0.711071,0.500000,...,1.000000,1.000000,0,1284,0,1580,0,322,0,395


## 13) Model comparison table

This table focuses on the metrics you will most likely discuss in the report.


In [30]:
comparison_cols = [
    'model_name',
    'cv_f1_mean_before_tuning',
    'best_cv_f1_after_tuning',
    'train_train_or_test_f1',
    'test_train_or_test_f1',
    'test_train_or_test_accuracy',
    'test_train_or_test_recall',
    'test_train_or_test_roc_auc',
    'test_success_precision',
    'test_missed_success_rate',
    'best_params',
]

results_df[comparison_cols]

,model_name,cv_f1_mean_before_tuning,best_cv_f1_after_tuning,train_train_or_test_f1,test_train_or_test_f1,test_train_or_test_accuracy,test_train_or_test_recall,test_train_or_test_roc_auc,test_success_precision,test_missed_success_rate,best_params
0,random_forest,0.754975,0.765248,1.000000,0.773109,0.736402,0.815190,0.793785,0.735160,0.184810,"{'model__max_depth': None, 'model__min_samples..."
1,logistic_regression,0.718765,0.723148,0.740604,0.728606,0.690377,0.754430,0.754438,0.704492,0.245570,"{'model__C': 0.1, 'model__solver': 'lbfgs'}"
2,svm_rbf,0.736250,0.736250,0.820352,0.728395,0.693166,0.746835,0.760064,0.710843,0.253165,"{'model__C': 1.0, 'model__gamma': 'scale'}"
3,decision_tree,0.664169,0.717518,0.733509,0.717949,0.662483,0.779747,0.717725,0.665227,0.220253,"{'model__max_depth': 4, 'model__min_samples_sp..."
4,baseline_dummy,0.711071,0.711071,0.711071,0.710432,0.550907,1.000000,0.500000,0.550907,0.000000,{'model__strategy': 'most_frequent'}


## 14) Best model summary


In [31]:
best_row = results_df.iloc[0]
best_model_name = best_row['model_name']
best_pipeline = trained_pipelines[best_model_name]

print('Best model:', best_model_name)
print('Best parameters:', best_row['best_params'])
best_row

Best model: random_forest
Best parameters: {'model__max_depth': None, 'model__min_samples_split': 2, 'model__n_estimators': 300}


model_name                                                           random_forest
cv_f1_mean_before_tuning                                                  0.754975
cv_f1_std_before_tuning                                                   0.007813
best_cv_f1_after_tuning                                                   0.765248
best_params                      {'model__max_depth': None, 'model__min_samples...
train_train_or_test_accuracy                                                   1.0
train_train_or_test_precision                                                  1.0
train_train_or_test_recall                                                     1.0
train_train_or_test_f1                                                         1.0
train_train_or_test_roc_auc                                                    1.0
train_success_precision                                                        1.0
train_missed_success_rate                                                      0.0
trai

## 15) Overfitting check

Comparing training and test F1 scores gives a quick view of overfitting.


In [32]:
overfit_view = results_df[['model_name', 'train_train_or_test_f1', 'test_train_or_test_f1']].copy()
overfit_view['f1_gap'] = overfit_view['train_train_or_test_f1'] - overfit_view['test_train_or_test_f1']
overfit_view.sort_values(by='f1_gap', ascending=False)

,model_name,train_train_or_test_f1,test_train_or_test_f1,f1_gap
0,random_forest,1.000000,0.773109,0.226891
2,svm_rbf,0.820352,0.728395,0.091957
3,decision_tree,0.733509,0.717949,0.015561
1,logistic_regression,0.740604,0.728606,0.011997
4,baseline_dummy,0.711071,0.710432,0.000639


## 16) Error analysis for the best model

We inspect false positives and false negatives, because they matter differently for the stakeholder.


In [33]:
test_analysis = X_test.copy()
test_analysis['actual'] = y_test.values
test_analysis['predicted'] = best_pipeline.predict(X_test)
test_analysis['predicted_proba'] = best_pipeline.predict_proba(X_test)[:, 1]

false_positives = test_analysis[(test_analysis['actual'] == 0) & (test_analysis['predicted'] == 1)].copy()
false_negatives = test_analysis[(test_analysis['actual'] == 1) & (test_analysis['predicted'] == 0)].copy()

print('False positives:', len(false_positives))
print('False negatives:', len(false_negatives))

False positives: 116
False negatives: 73


In [34]:
false_positives[['transfer_season', 'primary_position', 'age_at_transfer', 'market_value_at_transfer_eur', 'predicted_proba']].head(10)

,transfer_season,primary_position,age_at_transfer,market_value_at_transfer_eur,predicted_proba
557,23/24,Attack,20.407940,1225000.0,0.706667
1999,22/23,Midfield,22.234087,1225000.0,0.733333
1443,23/24,Midfield,24.555784,1225000.0,0.653333
1603,23/24,Attack,26.297056,1225000.0,0.593333
3303,22/23,Goalkeeper,34.288845,1225000.0,0.560000
3319,22/23,Attack,31.282682,1225000.0,0.560000
3195,22/23,Defender,28.221766,1225000.0,0.726667
2227,22/23,Defender,27.874060,1225000.0,0.576667
3276,22/23,Midfield,26.587269,1225000.0,0.750000
3127,22/23,Defender,20.873375,1225000.0,0.753333


In [35]:
false_negatives[['transfer_season', 'primary_position', 'age_at_transfer', 'market_value_at_transfer_eur', 'predicted_proba']].head(10)

,transfer_season,primary_position,age_at_transfer,market_value_at_transfer_eur,predicted_proba
2577,22/23,Goalkeeper,37.472965,1200000.0,0.243333
234,23/24,Attack,21.995893,1225000.0,0.376667
3158,22/23,Midfield,23.860369,300000.0,0.256667
225,23/24,Attack,21.894592,800000.0,0.396667
2433,22/23,Goalkeeper,31.750856,1225000.0,0.453333
1059,23/24,Defender,21.470226,1225000.0,0.230000
2097,22/23,Midfield,24.780287,450000.0,0.466667
2340,22/23,Midfield,19.485285,500000.0,0.233333
2510,22/23,Defender,24.002737,1225000.0,0.463333
370,23/24,Attack,21.582478,1225000.0,0.456667
